In [1]:
import sys
sys.path.append("../../")

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from NEW_PINN.PINN_const import EINN_PINN

# ============================================
# CONFIG
# ============================================

SAVE_DIR = "preliminary_experiments/Real_PINN_Results/"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_trials = 10

datasets = {
    "italy": {
        "path": "../real_datasets/PINN-COVID-Italy.csv",
        "train_size": 80,
        "init_params": {'beta': 0.1, 'gamma': 0.1, 'mu': 0.001},
        "lambda_ode": 0.1,
        "epochs": 10_000
    },
    "kouprianov": {
        "path": "../real_datasets/covid-19_Kouprianov.csv",
        "train_size": 185,
        "init_params": {'beta': 0.1, 'gamma': 0.1, 'mu': 0.001},
        "lambda_ode": 1.0,
        "epochs": 10_000
    }
}

# ============================================
# SEED
# ============================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================
# METRICS
# ============================================

def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

def mae(a, b):
    return np.mean(np.abs(a - b))


# ============================================
# SINGLE TRIAL
# ============================================

def run_trial(df, config, seed):

    set_seed(seed)

    S = df["S"].values
    I = df["I"].values
    R = df["R"].values
    D = df["D"].values

    t = np.arange(len(S)).astype(float)

    population = S[0] + I[0] + R[0] + D[0]

    model = EINN_PINN(
        t=t,
        S_data=S,
        I_data=I,
        R_data=R,
        D_data=D,
        population=population,
        train_size=config["train_size"],
        device=device,
        init_params=config["init_params"]
    )

    model.train_model(
        n_epoch=config["epochs"],
        lambda_data=1.0,
        lambda_ode=config["lambda_ode"],
        lambda_ic=0.1,
        lambda_bc=0.1
    )

    _, I_pred, _, _ = model.predict()
    I_pred = I_pred.numpy()

    # TEST part
    ts = config["train_size"]
    te = ts + 30

    I_true_test = I[ts:te]
    I_pred_test = I_pred[ts:te]

    # I metrics
    rmse_val = rmse(I_true_test, I_pred_test)
    mae_val = mae(I_true_test, I_pred_test)

    # PEAKS (REAL INTERPRETATION)
    true_peak_height = np.max(I_true_test)
    true_peak_time = np.argmax(I_true_test)

    pred_peak_height = np.max(I_pred_test)
    pred_peak_time = np.argmax(I_pred_test)

    params = model.params.get_params_dict()

    return {
        "rmse": rmse_val,
        "mae": mae_val,
        "true_peak_height": true_peak_height,
        "true_peak_time": true_peak_time,
        "pred_peak_height": pred_peak_height,
        "pred_peak_time": pred_peak_time,
        "beta": params["beta"],
        "gamma": params["gamma"],
        "mu": params["mu"],
        "I_pred": I_pred
    }


# ============================================
# EXPERIMENT
# ============================================

all_summaries = []

for name, cfg in datasets.items():

    print(f"\n================ {name.upper()} ================")

    df = pd.read_csv(cfg["path"])

    results = []
    all_preds = []

    for trial in range(n_trials):

        print(f"Trial {trial}")

        res = run_trial(df, cfg, seed=42 + trial)

        res["trial"] = trial
        results.append(res)
        all_preds.append(res["I_pred"])

    results_df = pd.DataFrame(results)

    # SAVE RAW
    results_df.to_csv(
        os.path.join(SAVE_DIR, f"r_d_{name}.csv"),
        index=False
    )

    # =========================
    # SUMMARY METRICS
    # =========================

    summary = {
        "dataset": name,

        "I RMSE (mean ± std)":
            f"{results_df['rmse'].mean():.4f} ± {results_df['rmse'].std():.4f}",

        "I MAE (mean ± std)":
            f"{results_df['mae'].mean():.4f} ± {results_df['mae'].std():.4f}",

        "Peak Height":
            f"{results_df['pred_peak_height'].mean():.4f}",

        "Peak Timing":
            f"{results_df['pred_peak_time'].mean():.2f}",

        "Peak Height (mean ± std)":
            f"{results_df['pred_peak_height'].mean():.4f} ± {results_df['pred_peak_height'].std():.4f}",

        "Peak Timing (mean ± std)":
            f"{results_df['pred_peak_time'].mean():.2f} ± {results_df['pred_peak_time'].std():.2f}",
    }

    all_summaries.append(summary)

    # =========================
    # PLOT I
    # =========================

    plt.figure(figsize=(10,6))

    t = np.arange(len(df))

    for p in all_preds:
        plt.plot(t, p, color="blue", alpha=0.3)

    plt.plot(df["I"].values, color="black", linewidth=2, label="True")

    plt.axvline(cfg["train_size"], color="red", linestyle="--")

    plt.title(f"I predictions - {name}")
    plt.legend()

    plt.savefig(os.path.join(SAVE_DIR, f"r_d_I_{name}.png"))
    plt.close()


# ============================================
# SAVE SUMMARY
# ============================================

summary_df = pd.DataFrame(all_summaries)

summary_df.to_csv(
    os.path.join(SAVE_DIR, "r_d_summary.csv"),
    index=False
)

print(summary_df)


================ ITALY ================
Trial 0


c:\Users\dinara\Desktop\НИР\Code\PINN_LLM\venv\Lib\site-packages\torch\optim\lr_scheduler.py:1340: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  current = float(metrics)


Epoch     0 | Loss: 3853945088.000000 | Data: 3675110912.000000 | ODE: 11063998.000000
Params: β=0.1000, γ=0.1000, μ=0.0010
---
Epoch  1000 | Loss: 28791410.000000 | Data: 26852514.000000 | ODE: 15884580.000000
Params: β=0.0852, γ=0.0837, μ=0.0009
---
Epoch  2000 | Loss: 15793699.000000 | Data: 14506335.000000 | ODE: 12873578.000000
Params: β=0.0751, γ=0.0716, μ=0.0012
---
Epoch  3000 | Loss: 13071848.000000 | Data: 12133747.000000 | ODE: 9380950.000000
Params: β=0.0659, γ=0.0618, μ=0.0016
---
Epoch  4000 | Loss: 9842204.000000 | Data: 9155033.000000 | ODE: 6871449.000000
Params: β=0.0583, γ=0.0540, μ=0.0020
---
Epoch  5000 | Loss: 5175951.000000 | Data: 4686843.000000 | ODE: 4890294.500000
Params: β=0.0522, γ=0.0478, μ=0.0024
---
Epoch  6000 | Loss: 2155335.250000 | Data: 1767261.625000 | ODE: 3880670.500000
Params: β=0.0471, γ=0.0425, μ=0.0030
---
Epoch  7000 | Loss: 1666019.125000 | Data: 1351730.500000 | ODE: 3142486.250000
Params: β=0.0426, γ=0.0377, μ=0.0035
---
Epoch  8000 | Los

In [9]:
import os
import pandas as pd
import numpy as np

SAVE_DIR = "preliminary_experiments/Real_PINN_Results/"

files = [
    "r_d_italy.csv",
    "r_d_kouprianov.csv"
]

# пути к оригинальным датасетам
real_datasets = {
    "italy": "../real_datasets/PINN-COVID-Italy.csv",
    "kouprianov": "../real_datasets/covid-19_Kouprianov.csv"
}

# train_size для каждого датасета
train_sizes = {
    "italy": 80,
    "kouprianov": 185
}

summary_rows = []

for file in files:

    df = pd.read_csv(os.path.join(SAVE_DIR, file))

    dataset = file.replace("r_d_", "").replace(".csv", "")

    # =============================
    # REAL PEAK (из оригинальных данных)
    # =============================
    real_df = pd.read_csv(real_datasets[dataset])

    I_true = real_df["I"].values

    # для Италии берем только первые 200 точек
    if dataset == "italy":
        I_true = I_true[:200]

    true_peak_height = np.max(I_true)
    true_peak_time = np.argmax(I_true)

    # =============================
    # FIX predicted peak timing
    # =============================
    train_size = train_sizes[dataset]

    pred_peak_time_mean = df['pred_peak_time'].mean() + train_size
    pred_peak_time_std = df['pred_peak_time'].std()

    summary = {

        "dataset": dataset,

        # I metrics
        "I RMSE (mean ± std)":
            f"{df['rmse'].mean():.4f} ± {df['rmse'].std():.4f}",

        "I MAE (mean ± std)":
            f"{df['mae'].mean():.4f} ± {df['mae'].std():.4f}",

        # REAL peaks
        "Peak Height":
            f"{true_peak_height:.4f}",

        "Peak Timing":
            f"{true_peak_time:.2f}",

        # predicted peaks
        "Peak Height (mean ± std)":
            f"{df['pred_peak_height'].mean():.4f} ± {df['pred_peak_height'].std():.4f}",

        "Peak Timing (mean ± std)":
            f"{pred_peak_time_mean:.2f} ± {pred_peak_time_std:.2f}",

        # эпид параметры
        "β predicted (mean ± std)":
            f"{df['beta'].mean():.4f} ± {df['beta'].std():.4f}",

        "γ predicted (mean ± std)":
            f"{df['gamma'].mean():.4f} ± {df['gamma'].std():.4f}",

        "μ predicted (mean ± std)":
            f"{df['mu'].mean():.6f} ± {df['mu'].std():.6f}",
    }

    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(SAVE_DIR, "r_d_summary.csv"),
    index=False
)

print(summary_df)

      dataset     I RMSE (mean ± std)      I MAE (mean ± std)  Peak Height  \
0       italy  13544.1227 ± 5923.5636  12301.5426 ± 5187.5048  108257.0000   
1  kouprianov  11790.9909 ± 1047.7098  11524.8780 ± 1019.7291  104932.0000   

  Peak Timing Peak Height (mean ± std) Peak Timing (mean ± std)  \
0       88.00   100223.4911 ± 630.3364             83.70 ± 1.34   
1      199.00   91766.9295 ± 1264.8608            190.30 ± 1.16   

  β predicted (mean ± std) γ predicted (mean ± std) μ predicted (mean ± std)  
0          0.0370 ± 0.0035          0.0277 ± 0.0013      0.006297 ± 0.000885  
1          0.0436 ± 0.0012          0.0333 ± 0.0011      0.004129 ± 0.000353  
